In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from datetime import datetime
from statsmodels.tsa.stattools import coint
import math

In [71]:
#config
start_date=datetime(2021,1,1)
end_date=datetime(2026,9,12)
okno = 63
liczba_sesji_w_roku = 252
#okno_sesji = 2*liczba_sesji_w_roku
okno_sesji = 126
bank_tickers = ["JPM", "BAC", "WFC", "C", "GS", "MS", "USB", "PNC", "TFC"]
tech_tickers = ["ORCL", "CRM", "ADBE", "CSCO", "IBM", "NOW", "INTC", "AMD", "QCOM", "TXN"]
all_tickers = bank_tickers + tech_tickers

In [3]:
df = yf.download(["KO", "PEP"], interval='1d', start=start_date, end=end_date)
df=df['Close']
df=df.reset_index()
df.columns.name = None
#Y = Pepsi, X=Cola
df

[*********************100%***********************]  2 of 2 completed


,Date,KO,PEP
0,2021-01-04,44.792770,119.999893
1,2021-01-05,44.300362,120.357567
2,2021-01-06,42.891037,118.885292
3,2021-01-07,42.415607,118.502724
4,2021-01-08,43.366474,119.925087
...,...,...,...
1424,2026-09-04,88.070000,137.630005
1425,2026-09-08,88.360001,138.449997
1426,2026-09-09,87.550003,136.690002
1427,2026-09-10,87.830002,136.649994


In [4]:
X = np.log(df["KO"])
Y = np.log(df["PEP"])
X.head()

0    3.802047
1    3.790993
2    3.758663
3    3.747516
4    3.769687
Name: KO, dtype: float64

In [73]:
ones = np.ones(okno_sesji)
X = X[:okno_sesji]
X_mat = np.column_stack((ones,X))
Y = Y[:okno_sesji]
B = np.linalg.inv(X_mat.T @ X_mat) @ X_mat.T @ Y
alpha = B[0]
beta = B[1]

In [75]:
print(f"alpha: {alpha}, beta: {beta}")

alpha: 7.477939042519276, beta: -0.5273036137694385


In [77]:
Y_pred = beta*X + alpha
residuals = Y - Y_pred
residuals.mean()

np.float64(-3.089733054944118e-12)

In [79]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.adfvalues import mackinnonp
# 2.ADF 
adf_res = adfuller(residuals, regression='n', autolag='AIC') # W adfuller dajemy 'n', bo reszty z OLS mają już z definicji średnią zero (brak stałej w ADF),
t_stat = adf_res[0]

# 3.Wyciągnięcie dokładnego p-value do FDR
p_val = mackinnonp(t_stat, regression='c', N=2) # a w mackinnonp 'c', bo tablice kointegracji muszą wiedzieć, że pierwotna regresja cenowa zawierała stałą.
p_val

/tmp/ipykernel_716/2530293827.py:4: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adf_res = adfuller(residuals, regression='n', autolag='AIC') # W adfuller dajemy 'n', bo reszty z OLS mają już z definicji średnią zero (brak stałej w ADF),


0.20755619252264157

In [81]:
coint_T, pval, crit_values = coint(Y, X)
pval

0.20755619252602076

# Częśc druga

In [83]:
df = yf.download(bank_tickers, interval='1d', start=start_date, end=end_date)
df=df['Close']
df=df.reset_index()
df.columns.name = None
df.dropna(inplace=True)

[*********************100%***********************]  9 of 9 completed


In [85]:
df_ceny=df.drop(columns='Date')
df_ceny_log=np.log(df_ceny)
df_ceny_log_shifted=df_ceny_log.shift(1)
zwroty=df_ceny_log-df_ceny_log_shifted
zwroty=zwroty.dropna()
korelacja=zwroty.corr()
zwroty

,BAC,C,GS,JPM,MS,PNC,TFC,USB,WFC
1,0.007630,0.025609,0.022131,0.005427,0.016834,-0.005310,0.006760,0.003247,0.027563
2,0.060586,0.055942,0.052556,0.045887,0.058549,0.053478,0.051293,0.057530,0.068359
3,0.021843,0.011732,0.021138,0.032311,0.016956,0.015784,0.029365,0.019799,0.022384
4,-0.010093,-0.009894,-0.005398,0.001103,0.004128,-0.005107,-0.016449,-0.013493,-0.007205
5,0.016464,0.016236,0.012845,0.014814,0.010707,0.007494,0.005709,0.000405,0.001505
...,...,...,...,...,...,...,...,...,...
1424,-0.000638,-0.003045,0.000655,-0.009491,0.002622,0.001712,0.000775,-0.002522,0.008707
1425,-0.004637,-0.007141,-0.002005,-0.014407,-0.006821,-0.011013,-0.021527,-0.013984,-0.022594
1426,0.004478,0.007722,-0.007505,0.003389,-0.004124,-0.009181,-0.007944,-0.006100,0.019254
1427,-0.001757,0.005067,-0.008797,-0.003247,-0.012570,0.009469,0.003583,0.004819,-0.002456


In [12]:
def dwumian_newtona(n:int,k:int):
    return int(math.factorial(n)/(math.factorial(n-k)*math.factorial(k)))

In [93]:
from itertools import permutations
wszystkie_slowniki=[]
for para in permutations(bank_tickers,2):
    X = np.log(df[para[0]])
    Y = np.log(df[para[1]])
    ones = np.ones(okno_sesji)
    X = X[:okno_sesji]
    X_mat = np.column_stack((ones,X))
    Y = Y[:okno_sesji]
    B = np.linalg.inv(X_mat.T @ X_mat) @ X_mat.T @ Y
    alpha = B[0]
    beta = B[1]
    Y_pred = beta*X + alpha
    residuals = Y - Y_pred
    adf_res = adfuller(residuals, regression='n', autolag='AIC', result_object=False)
    t_stat = adf_res[0]
    p_val = mackinnonp(t_stat, regression='c', N=2)
    pojedynczy_slownik={
        'objasniajaca': para[0],
        'objasniana': para[1],
        'pval': p_val,
        'alpha': alpha,
        'beta': beta
    }
    wszystkie_slowniki.append(pojedynczy_slownik)

In [95]:
dobre_slowniki=[]
for slownik in wszystkie_slowniki:
    if slownik.get('pval')<=0.05:
        dobre_slowniki.append(slownik)

len(wszystkie_slowniki)

72

In [97]:
def FDR_test(slownik, q = 0.05, m: int = None) :
    #logika czy kolumna pval istnieej
    df = pd.DataFrame(slownik)
    df.sort_values("pval",ascending = True, inplace=True)
    df.reset_index(drop=True, inplace=True)
    df['rank'] = df.index + 1
    df['FDR'] = (df['rank'] / m) * q
    df['is_significant'] = df['pval'] <= df['FDR']
    significant_ranks = df.loc[df["is_significant"], "rank"]

    if not significant_ranks.empty:
        max_k = significant_ranks.max()
        df["reject_H0"] = df["rank"] <= max_k
    else:
        df['reject_H0'] = False

    return df

In [113]:
super_df = FDR_test(wszystkie_slowniki,q=0.1,m=len(wszystkie_slowniki))
super_df

,objasniajaca,objasniana,pval,alpha,beta,rank,FDR,is_significant,reject_H0
0,USB,JPM,0.035376,2.504612,0.630973,1,0.001389,False,False
1,PNC,GS,0.052791,0.373976,1.066437,2,0.002778,False,False
2,MS,GS,0.059457,0.863754,1.139856,3,0.004167,False,False
3,C,JPM,0.062134,1.550510,0.818255,4,0.005556,False,False
4,WFC,PNC,0.062697,2.794805,0.613661,5,0.006944,False,False
...,...,...,...,...,...,...,...,...,...
67,MS,C,0.965512,0.415523,0.864129,68,0.094444,False,False
68,TFC,PNC,0.972155,0.804255,1.103504,69,0.095833,False,False
69,TFC,BAC,0.978848,-1.540300,1.331846,70,0.097222,False,False
70,C,MS,0.979005,0.527799,0.909277,71,0.098611,False,False


In [ ]:
#Half life by byl gdyby byla jakas kointegracja :C a nie ma 

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=6266d961-fef7-4a47-b36c-beb13dd80930' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>